In [1]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

In [4]:
df = pd.read_excel('df469.xlsx')
df.rename(columns={'модуль упругости, ГПа':'Mодуль упругости, ГПа'}, inplace=True)

In [5]:
# Фиксация сидов для строгого сравнения на одинаковых условиях
torch.manual_seed(42)
np.random.seed(42)

In [6]:
# 1. ПОДГОТОВКА ДАННЫХ И ИСПРАВЛЕНИЕ ИМЕН КОЛОНОК
# -------------------------------------------------------------------
feature_cols = [
    'Плотность, кг/м3', 'Mодуль упругости, ГПа', 'Количество отвердителя, м.%',
    'Содержание эпоксидных групп,%_2', 'Температура вспышки, С_2', 'Поверхностная плотность, г/м2',
    'Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа',
    'Потребление смолы, г/м2', 'Шаг нашивки', 'Плотность нашивки'
]
target_col = 'Соотношение матрица-наполнитель'


X = df[feature_cols].values
y = df[target_col].values.reshape(-1, 1)


In [7]:
# 2. АРХИТЕКТУРА НЕЙРОННОЙ СЕТИ (PyTorch)
# --------------------------------------------------------------------
class CompositeMLP(nn.Module):
    def __init__(self, input_dim):
        super(CompositeMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.35), # Защита от переобучения

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.network(x)

In [8]:
# 3. КРОСС-ВАЛИДАЦИЯ И СРАВНЕНИЕ МОДЕЛЕЙ
# ---------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Списки для хранения метрик по каждому фолду
nn_metrics = {'r2': [], 'mae': [], 'rmse': []}
cb_metrics = {'r2': [], 'mae': [], 'rmse': []}

print(f"Запуск кросс-валидации на {len(df)} строках...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    # Разделение данных
    X_train_f, X_val_f = X[train_idx], X[val_idx]
    y_train_f, y_val_f = y[train_idx], y[val_idx]

    # Масштабирование (Важно для ИНС, для CatBoost не обязательно, но оставим для идентичности X)
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_f)
    X_val_scaled = scaler_X.transform(X_val_f)

    y_train_scaled = scaler_y.fit_transform(y_train_f)
    y_val_scaled = scaler_y.transform(y_val_f)

Запуск кросс-валидации на 469 строках...



In [9]:

    # ----------- ЧАСТЬ 1: ОБУЧЕНИЕ НЕЙРОСЕТИ (PyTorch) -----------
    X_train_t = torch.FloatTensor(X_train_scaled)
    y_train_t = torch.FloatTensor(y_train_scaled)
    X_val_t = torch.FloatTensor(X_val_scaled)
    y_val_t = torch.FloatTensor(y_val_scaled)

    nn_model = CompositeMLP(input_dim=len(feature_cols))
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(nn_model.parameters(), lr=0.01, weight_decay=1e-3) # L2 регуляризация

    best_val_loss = float('inf')
    patience, patience_counter = 15, 0
    best_model_state = None

    for epoch in range(200):
        nn_model.train()
        optimizer.zero_grad()
        loss = criterion(nn_model(X_train_t), y_train_t)
        loss.backward()
        optimizer.step()

        nn_model.eval()
        with torch.no_grad():
            val_loss = criterion(nn_model(X_val_t), y_val_t).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = nn_model.state_dict().copy()
        else:
            patience_counter += 1
        if patience_counter >= patience:
            break

    nn_model.load_state_dict(best_model_state)
    nn_model.eval()
    with torch.no_grad():
        nn_preds_scaled = nn_model(X_val_t).numpy()
        nn_preds = scaler_y.inverse_transform(nn_preds_scaled)

In [10]:
    # ----------- ЧАСТЬ 2: ОБУЧЕНИЕ CATBOOST -----------
    # Настраиваем параметры под малую выборку
    cb_model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.03,
        depth=4,                  # Неглубокие деревья, чтобы не переобучаться
        l2_leaf_reg=5,            # Регуляризация листьев
        random_seed=42,
        verbose=False             # Отключаем вывод логов обучения
    )

    # Обучаем CatBoost на немасштабированном Y (он сам прекрасно с ним работает)
    cb_model.fit(X_train_scaled, y_train_f.ravel(), eval_set=(X_val_scaled, y_val_f.ravel()), early_stopping_rounds=30)
    cb_preds = cb_model.predict(X_val_scaled)

In [11]:
    # ----------- СБОР МЕТРИК -----------
    # Нейросеть
    nn_metrics['r2'].append(r2_score(y_val_f, nn_preds))
    nn_metrics['mae'].append(mean_absolute_error(y_val_f, nn_preds))
    nn_metrics['rmse'].append(np.sqrt(mean_squared_error(y_val_f, nn_preds)))

    # Бустинг
    cb_metrics['r2'].append(r2_score(y_val_f, cb_preds))
    cb_metrics['mae'].append(mean_absolute_error(y_val_f, cb_preds))
    cb_metrics['rmse'].append(np.sqrt(mean_squared_error(y_val_f, cb_preds)))

    print(f"Фолд {fold+1} | ИНС R²: {nn_metrics['r2'][-1]:.3f} vs CatBoost R²: {cb_metrics['r2'][-1]:.3f}")

Фолд 5 | ИНС R²: -0.120 vs CatBoost R²: 0.007


In [12]:

# 4. СРАВНИТЕЛЬНЫЙ АНАЛИЗ (ИТОГИ)
# -----------------------------------------------------------
results_df = pd.DataFrame({
    'Метрика': ['MAE', 'RMSE', 'R²'],
    'Нейросеть (PyTorch)': [np.mean(nn_metrics['mae']), np.mean(nn_metrics['rmse']), np.mean(nn_metrics['r2'])],
    'CatBoost': [np.mean(cb_metrics['mae']), np.mean(cb_metrics['rmse']), np.mean(cb_metrics['r2'])]
}).set_index('Метрика')

print("_"*55)
print("СРАВНЕНИЕ МЕТРИК:СРЕДНИЕ ЗНАЧЕНИЯ ПО КРОСС-ВАЛИДАЦИИ")
print("_"*55)
print(results_df.round(4))



_______________________________________________________
СРАВНЕНИЕ МЕТРИК:СРЕДНИЕ ЗНАЧЕНИЯ ПО КРОСС-ВАЛИДАЦИИ
_______________________________________________________
         Нейросеть (PyTorch)  CatBoost
Метрика                               
MAE                   0.6660    0.6285
RMSE                  0.8259    0.7776
R²                   -0.1200    0.0071


In [14]:
# 5. Сохранение файлов для GUI-приложения
MODEL_FILE = 'base_catboost_model.cbm'
cb_model.save_model(MODEL_FILE)
print(f"\nБазовая модель CatBoost успешно сохранена в файл '{MODEL_FILE}'")

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Скейлер признаков успешно сохранен в файл 'scaler.pkl'")


Базовая модель CatBoost успешно сохранена в файл 'base_catboost_model.cbm'


NameError: name 'pickle' is not defined